# clip_stacker — RIFE on Google Colab

Run the same **Practical-RIFE v4.26** pipeline that backs the Hugging Face Space
[`1inkusFace/RIFE`](https://huggingface.co/spaces/1inkusFace/RIFE)
(`huggingface/RIFE/` in [ford442/clip_stacker](https://github.com/ford442/clip_stacker)).

| Space / client contract | This notebook |
|---|---|
| `interpolate_video` | single-clip interpolate + optional boomerang |
| multiplier `"2"` / `"4"` / `"8"` | same |
| output fps `"30"` / `"60"` / `"native"` | same meaning as `app.py` |
| `stitch` / `morph` / batch | not here — those stay on the Space |

RIFE always generates `source_fps × multiplier` frames. `OUTPUT_FPS` only
controls the *encode*: `60` is the point of 4× on a 24 fps source; `30` matches
the Space's old default; `native` keeps every generated frame.

**Runtime → Change runtime type → T4 / L4 / A100 GPU.** CPU works but is slow.


In [ ]:
# GPU check — Runtime > Change runtime type > GPU if this says CPU
import torch, subprocess, shutil, sys
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("ffmpeg", shutil.which("ffmpeg"))
print("ffprobe", shutil.which("ffprobe"))


## 1. Install + clone Practical-RIFE + v4.26 weights

Same sources as `huggingface/RIFE/app.py` → `setup_environment()`:
- code: `https://github.com/hzwer/Practical-RIFE`
- weights: `https://huggingface.co/hzwer/RIFE/resolve/main/RIFEv4.26_0921.zip`


In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

BASE = Path("/content")
RIFE_DIR = BASE / "Practical-RIFE"
WORKSPACE = BASE / "workspace_temp"
MODEL_URL = "https://huggingface.co/hzwer/RIFE/resolve/main/RIFEv4.26_0921.zip"
STITCH_FPS = 30
VIDEO_TRACK_TIMESCALE = 30000

WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.check_call(cmd, shell=isinstance(cmd, str), **kw)

# Colab already has a CUDA torch; do not pip-install a CPU wheel over it.
run([sys.executable, "-m", "pip", "install", "-q",
     "opencv-python>=4.1.2", "tqdm", "requests", "huggingface_hub"])

if not (RIFE_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "https://github.com/hzwer/Practical-RIFE", str(RIFE_DIR)])

hdv3 = RIFE_DIR / "HDv3"
train_log = RIFE_DIR / "train_log"
need_weights = not hdv3.exists() or not (train_log / "RIFE_HDv3.py").exists()

if need_weights:
    zip_path = RIFE_DIR / "RIFEv4.26_0921.zip"
    run(["wget", "-q", "-O", str(zip_path), MODEL_URL])
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RIFE_DIR)
    extract = RIFE_DIR / "RIFEv4.26_0921"
    train_log.mkdir(exist_ok=True)
    for name in ("RIFE_HDv3.py", "IFNet_HDv3.py"):
        src = extract / name
        if src.exists():
            shutil.move(str(src), str(train_log / name))
    (train_log / "__init__.py").write_text("")
    hdv3.mkdir(exist_ok=True)
    flownet = extract / "flownet.pkl"
    if flownet.exists():
        shutil.move(str(flownet), str(hdv3 / "flownet.pkl"))
    shutil.rmtree(extract, ignore_errors=True)
    zip_path.unlink(missing_ok=True)

# Space patches mpeg4 → libx264 so inference_video.py writes an editor-friendly MP4.
inf = RIFE_DIR / "inference_video.py"
text = inf.read_text()
old = "-c:v', 'mpeg4', '-qscale:v', '1'"
new = "-c:v', 'libx264', '-preset', 'medium', '-crf', '23'"
if old in text and "libx264" not in text:
    inf.write_text(text.replace(old, new))
    print("patched inference_video.py codec → libx264")
else:
    print("inference_video.py codec patch skipped (already present or upstream changed)")

print("RIFE_DIR", RIFE_DIR)
print("weights", list((RIFE_DIR / "HDv3").glob("*")))
print("train_log", list(train_log.glob("*.py")))


## 2. Encode helpers (same contract as `app.py`)

Copied in spirit from `nle_friendly_h264_video_args` / boomerang / probe helpers
so Colab output matches what clip_stacker expects from the Space.


In [ ]:
import uuid

def nle_friendly_h264_video_args(gop=None, fps=None):
    is_native = fps == "native"
    fps = STITCH_FPS if fps is None or is_native else fps
    fps_val = float(fps)
    fps_str = str(int(fps_val)) if fps_val.is_integer() else str(fps_val)
    gop = gop or int(fps_val)
    args = [
        "-c:v", "libx264", "-preset", "fast", "-crf", "18",
        "-tune", "stillimage",
        "-profile:v", "high", "-level", "4.1",
        "-g", str(gop), "-keyint_min", str(gop), "-sc_threshold", "0",
        "-pix_fmt", "yuv420p",
        "-bf", "0",
    ]
    if not is_native:
        args += ["-vsync", "cfr", "-r", fps_str]
    args += [
        "-colorspace", "bt709", "-color_primaries", "bt709",
        "-color_trc", "bt709", "-color_range", "tv",
        "-video_track_timescale", str(VIDEO_TRACK_TIMESCALE),
    ]
    return args

def nle_friendly_aac_audio_args():
    return ["-c:a", "aac", "-b:a", "192k", "-ar", "44100", "-ac", "2"]

def get_fps(path):
    try:
        out = subprocess.check_output([
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "stream=avg_frame_rate",
            "-of", "default=noprint_wrappers=1:nokey=1", path,
        ], text=True).strip()
        if "/" in out:
            n, d = map(int, out.split("/"))
            return n / d if d else 30.0
        return float(out)
    except Exception:
        return 30.0

def get_duration(path):
    try:
        out = subprocess.check_output([
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1", path,
        ], text=True).strip()
        return float(out)
    except Exception:
        return None

def probe_has_audio(path):
    try:
        out = subprocess.check_output([
            "ffprobe", "-v", "error", "-select_streams", "a:0",
            "-show_entries", "stream=codec_name",
            "-of", "default=noprint_wrappers=1:nokey=1", path,
        ], text=True).strip()
        return bool(out)
    except Exception:
        return False

def rife_timeout_seconds(path, multi):
    duration = get_duration(path) or 60.0
    try:
        multi_val = max(1, int(str(multi).strip().replace("x", "") or "2"))
    except ValueError:
        multi_val = 2
    return max(300, int(duration * multi_val * 45 + 120))

def create_boomerang_loop(input_path, output_path, fps):
    sid = uuid.uuid4().hex
    rev = WORKSPACE / f"boom_rev_{sid}.mp4"
    fwd = WORKSPACE / f"boom_fwd_{sid}.mp4"
    lst = WORKSPACE / f"boom_list_{sid}.txt"
    enc = ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", str(fps),
           "-preset", "slow", "-crf", "17", "-an"]
    try:
        subprocess.run(["ffmpeg", "-i", input_path, "-vf", "reverse", *enc, "-y", str(rev)],
                       check=True, capture_output=True, text=True)
        subprocess.run(["ffmpeg", "-i", input_path, *enc, "-y", str(fwd)],
                       check=True, capture_output=True, text=True)
        lst.write_text(f"file '{fwd}'\nfile '{rev}'\n")
        subprocess.run(["ffmpeg", "-f", "concat", "-safe", "0", "-i", str(lst),
                        "-c", "copy", "-movflags", "+faststart", "-y", output_path],
                       check=True, capture_output=True, text=True)
        return True
    except subprocess.CalledProcessError as e:
        print("Boomerang failed:", e.stderr or e)
        return False
    finally:
        for p in (rev, fwd, lst):
            p.unlink(missing_ok=True)

def run_rife_inference(video_path, output_path, multi):
    timeout = rife_timeout_seconds(video_path, multi)
    return subprocess.run(
        [sys.executable, "inference_video.py",
         "--video", video_path, "--output", output_path,
         "--multi", str(multi), "--model", "HDv3"],
        capture_output=True, text=True, cwd=str(RIFE_DIR), timeout=timeout,
    )

print("helpers ready")


## 3. Upload a clip

Use the file picker, or set `INPUT_PATH` to a file already on disk
(`/content/drive/MyDrive/...` after mounting Drive).


In [ ]:
from google.colab import files, drive
from pathlib import Path

# Optional: mount Drive once if you want to read/write there
MOUNT_DRIVE = False
if MOUNT_DRIVE and not Path("/content/drive").exists():
    drive.mount("/content/drive")

# Leave empty to open the Colab upload widget
INPUT_PATH = ""

if not INPUT_PATH:
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    name = next(iter(uploaded))
    dest = WORKSPACE / name
    dest.write_bytes(uploaded[name])
    INPUT_PATH = str(dest)

INPUT_PATH = str(Path(INPUT_PATH))
assert Path(INPUT_PATH).exists(), INPUT_PATH
print("input:", INPUT_PATH)
print("duration:", get_duration(INPUT_PATH), "s  fps:", get_fps(INPUT_PATH))


## 4. Interpolate

Same knobs as the Space `interpolate_video` route / clip_stacker client.


In [ ]:
# --- knobs (match huggingface/RIFE README contract) ---
MULTI = 4            # 2 | 4 | 8
OUTPUT_FPS = "60"    # "30" | "60" | "native"
BOOMERANG = False
# ------------------------------------------------------

factor = str(MULTI).strip().replace("x", "")
fps_arg = str(OUTPUT_FPS).strip().lower()
fps_arg = "native" if fps_arg == "native" else float(fps_arg)

sid = uuid.uuid4().hex
safe_input = WORKSPACE / f"input_{sid}.mp4"
rife_out = WORKSPACE / f"output_rife_{sid}.mp4"
final_path = WORKSPACE / f"final_interp_{sid}.mp4"
boom_path = WORKSPACE / f"final_boom_{sid}.mp4"
no_audio = Path(str(rife_out).replace(".mp4", "_noaudio.mp4"))

shutil.copy(INPUT_PATH, safe_input)
print(f"RIFE {factor}x → {OUTPUT_FPS}  timeout={rife_timeout_seconds(str(safe_input), factor)}s")

r = run_rife_inference(str(safe_input), str(rife_out), factor)
if r.returncode != 0:
    print(r.stdout)
    raise RuntimeError(f"RIFE failed:\n{r.stderr[-4000:]}")

src = rife_out if rife_out.exists() else no_audio
if not src.exists():
    raise FileNotFoundError(f"RIFE wrote nothing. stdout:\n{r.stdout}\nstderr:\n{r.stderr}")

cmd = ["ffmpeg", "-i", str(src)]
if probe_has_audio(str(safe_input)):
    cmd += ["-i", str(safe_input), "-map", "0:v:0", "-map", "1:a:0?",
            *nle_friendly_h264_video_args(fps=fps_arg),
            *nle_friendly_aac_audio_args(),
            "-movflags", "+faststart", "-shortest", "-y", str(final_path)]
else:
    cmd += [*nle_friendly_h264_video_args(fps=fps_arg),
            "-an", "-movflags", "+faststart", "-y", str(final_path)]
subprocess.run(cmd, check=True, capture_output=True, text=True)

out = final_path
if BOOMERANG:
    if create_boomerang_loop(str(final_path), str(boom_path), get_fps(str(final_path))):
        out = boom_path

OUTPUT_PATH = str(out)
print("output:", OUTPUT_PATH)
print("duration:", get_duration(OUTPUT_PATH), "s  fps:", get_fps(OUTPUT_PATH),
      "size_mb:", round(Path(OUTPUT_PATH).stat().st_size / 1e6, 2))


## 5. Preview + download

In [ ]:
from IPython.display import Video, display
from google.colab import files

display(Video(OUTPUT_PATH, embed=True, width=640))
files.download(OUTPUT_PATH)


## Notes

- Do **not** add a second RIFE implementation under `src/`. The Space at
  `huggingface/RIFE/` is the source of truth; this notebook is a Colab runner
  for the same weights + `inference_video.py` path.
- `@spaces.GPU` / Gradio / ZeroGPU token chunking do not apply on Colab.
- Long clips + 8× will burn the Colab session. Start with a few seconds at 2×.
- If `inference_video.py` upstream changes flags, update the subprocess argv
  here and in `run_rife_inference()` in `huggingface/RIFE/app.py` together.
